# 1. 讀取檔案

In [1]:
!pip install pandas
!pip install scikit-learn
!pip install numpy
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
# 引入更強大的集成模型來衝高分數、過 Medium Baseline
from sklearn.ensemble import RandomForestRegressor 

# 使用 pandas 的 read_csv 函式讀取訓練資料
df = pd.read_csv("data/train.csv")
print("訓練資料讀取成功，形狀為：", df.shape)


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
訓練資料讀取成功，形狀為： (318, 10)


# 2.資料預處理

In [2]:
# 捨棄直接 dropna() 的粗暴作法，改用各欄位的中位數填補訓練集的缺失值
# 這樣可以保留更多訓練樣本，且能應對測試集可能出現的缺失值
num_cols = ['weight', 'acceleration', 'model_year', 'cylinders', 'displacement', 'horsepower']
medians = df[num_cols].median()

df_clean = df.copy()
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(medians[col])

print("訓練集缺失值填補完成！")

訓練集缺失值填補完成！


# 3.特徵工程

In [3]:
def create_features(input_df):
    """
    特徵工程函式：建立衍生特徵以提升模型預測實力
    """
    output_df = input_df.copy()
    
    # 1. 馬力重量比 (Power-to-Weight Ratio) - 影響油耗的關鍵物理指標
    output_df['power_to_weight'] = output_df['horsepower'] / (output_df['weight'] + 1e-5)
    
    # 2. 每缸排氣量 (Displacement per Cylinder)
    output_df['displacement_per_cylinder'] = output_df['displacement'] / (output_df['cylinders'] + 1e-5)
    
    # 3. 加速與馬力比
    output_df['acc_hp_ratio'] = output_df['acceleration'] / (output_df['horsepower'] + 1e-5)
    
    return output_df

# 套用特徵工程
df_featured = create_features(df_clean)

# 定義最終要送入模型訓練的特徵欄位（包含新建立的衍生特徵）
features = [
    'weight', 'acceleration', 'model_year', 'cylinders', 'displacement', 'horsepower',
    'power_to_weight', 'displacement_per_cylinder', 'acc_hp_ratio'
]
target = 'mpg'

# 選取 X 和 y
X = df_featured[features]
y = df_featured[target]

print("特徵工程建立完成，目前的特徵包含：", features)

特徵工程建立完成，目前的特徵包含： ['weight', 'acceleration', 'model_year', 'cylinders', 'displacement', 'horsepower', 'power_to_weight', 'displacement_per_cylinder', 'acc_hp_ratio']


# 4. 訓練模型

In [4]:
# 分割訓練集與測試集 (用於本地評估模型好壞)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 建立隨機森林回歸模型 (調整參數以防過擬合並提升泛化能力)
model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# 進行預測與本地評估 (指標為 RMSE)
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)
train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print("模型已訓練完成！")
print(f"本地訓練誤差 (Train RMSE): {train_rmse:.4f} MPG")
print(f"本地驗證誤差 (Test RMSE):  {test_rmse:.4f} MPG")
print("\n💡 提示：若 Test RMSE 顯著低於原本線性回歸的 3.85，代表極有機會通過 Medium Baseline！")

print("\n--- 特徵重要性 (Feature Importances) ---")
# 隨機森林提供特徵重要性評估
for feat, importance in zip(features, model.feature_importances_):
    print(f"特徵 '{feat}' 的重要性分數: {importance:.4f}")

模型已訓練完成！
本地訓練誤差 (Train RMSE): 1.1151 MPG
本地驗證誤差 (Test RMSE):  2.9853 MPG

💡 提示：若 Test RMSE 顯著低於原本線性回歸的 3.85，代表極有機會通過 Medium Baseline！

--- 特徵重要性 (Feature Importances) ---
特徵 'weight' 的重要性分數: 0.1814
特徵 'acceleration' 的重要性分數: 0.0139
特徵 'model_year' 的重要性分數: 0.1182
特徵 'cylinders' 的重要性分數: 0.1965
特徵 'displacement' 的重要性分數: 0.2708
特徵 'horsepower' 的重要性分數: 0.1521
特徵 'power_to_weight' 的重要性分數: 0.0155
特徵 'displacement_per_cylinder' 的重要性分數: 0.0317
特徵 'acc_hp_ratio' 的重要性分數: 0.0200


# 5.輸出提交檔案

In [5]:
# 讀取需要進行預測的測試檔案 test.csv
df_test = pd.read_csv("data/test.csv")

# 對測試資料進行相同的特徵工程與缺失值填補處理
df_test_clean = df_test.copy()
for col in num_cols:
    # 使用與訓練集相同的特徵中位數進行填補，避免 Data Leakage
    df_test_clean[col] = df_test_clean[col].fillna(medians[col])

# 套用測試集的特徵工程
df_test_featured = create_features(df_test_clean)

# 使用訓練好的模型，對測試資料進行預測
predictions = model.predict(df_test_featured[features])

# 建立符合 Kaggle 規範格式的提交 DataFrame
submission_df = pd.DataFrame({'Id': df_test['id'], 'mpg': predictions})

# 保存為 submission.csv
submission_df.to_csv('submission.csv', index=False)
print("提交文件 'submission.csv' 已成功生成！快去 Kaggle 上傳測評吧 🚀")

提交文件 'submission.csv' 已成功生成！快去 Kaggle 上傳測評吧 🚀


# 6. 報告

姓名：李思妤 學號：113403529

第一部分：準確度分數 (Accuracy Scores) (1分)  
我的準確度分數： 2.39404

第二部分：我的實驗記錄 (My Experiment Log) (3分)  
請記錄你做了哪些嘗試來提升分數，至少記錄兩次不同的嘗試。  

【實驗 1】  
    我做的修改：將缺失值處理從直接刪除（dropna）改為使用中位數（Median）填補；加入衍生特徵（馬力重量比 hp_per_weight、每汽缸平均排氣量 disp_per_cyl），並將漏掉的產地特徵（origin）進行 One-Hot 編碼；模型從簡單線性回歸更換為隨機森林回歸模型（RandomForestRegressor），將 train/test 特徵合併處理以防測試集對齊出錯。 

    結果與觀察 (分數變化、心得等)：改用非線性的隨機森林並加入物理交互特徵後，誤差明顯下降，Kaggle 盲測分數也來到 2.46968。這證明了對於結構化車輛資料，樹模型能比線性模型更好地捕捉特徵間複雜的非線性關係，且補齊產地類別變數對預測有顯著幫助。  

    該次實驗分數： 2.46968  

【實驗 2】  
    我做的修改：在實驗 1 的基礎上進一步精簡特徵工程與處理流程。在特徵部分新增了加速與馬力比（acc_hp_ratio），並將特徵名稱統一微調優化（如 power_to_weight、displacement_per_cylinder），同時維持隨機森林回歸器（參數設定樹木總數 200、最大深度 10）進行訓練，穩定模型的泛化能力。 

    結果與觀察 (分數變化、心得等)：經過特徵微調與交互變數的調整，Kaggle 預測分數進一步優化，成功下降至 2.39404。這讓我體會到特徵工程的細微調整對模型預測精度有直接影響，細緻地定義物理實體指標，能讓機器學習模型更準確地預估汽車油耗。  

    該次實驗分數： 2.39404

第三部分：總結與心得 (Conclusion & Reflection) (2分)  
請撰寫一段約 50-100 字的心得總結。內容需包含：  
(1) 你認為本次實驗中，提升準確率最有效的修改是什麼。  
(2) 這次不斷嘗試與修正的過程，帶給你最大的學習與啟發。  
內容：本次實驗中，提升準確率最有效的修改是將線性回歸更換為隨機森林模型，並搭配中位數填補與創造交互特徵（如馬力重量比）。這次不斷嘗試與修正的過程，讓我深刻體會到「資料與特徵決定了機器學習的上限，而模型只是逼近這個上限」。盲目套用模型而不做前處理只會適得其反，唯有理解數據背後的物理意義並耐心調整，才能真正實現優化。 